# 5 · Versions, a package of nodes, the palette

- A newer version names the one it replaces, a `@node` of the same id. The decorated function is the newest. `Upgrade` moves a saved graph on, and `Deprecation` marks a version or a node as going away.
- Each module keeps a registry of its own, and the host adds it with `add_subregistry`.
- `registry.describe()` is the record a frontend renders, and each node in it a `NodeDescription`, the same record for a function and a subgraph; `model_dump(mode=\"json\")` writes it as JSON.

In [ ]:
from typing import Annotated

from conductor import Deprecation, NodeRegistry, Param, Upgrade, node
from conductor.widgets import TextInput


@node("echo", title="Echo", description="Hands back the text it is given.", deprecation=Deprecation(header="Use version 2", migration="Version 2 adds a prefix; existing graphs keep working."))
def echo_v1(text: Annotated[str, Param(title="Input", widget=TextInput())]) -> Annotated[str, Param(title="Output")]:
    return text


@node("echo", title="Echo", description="Hands back the text, with a prefix in front if you give one.", category="text", replaces=echo_v1, upgrade=Upgrade())
def echo(
    text: Annotated[str, Param(title="Input", widget=TextInput())],
    prefix: Annotated[str, Param(widget=TextInput())] = "",
) -> Annotated[str, Param(title="Output")]:
    return f"{prefix}{text}"


registry = NodeRegistry()
registry.add(echo)


## Versions

A graph pinned at version 1 runs version 1. A deprecation is a notice, not an error.

In [ ]:
print("versions:      ", echo.versions)
print("current:       ", echo.version)
print("v1 deprecated? ", echo.version_of(1).deprecation)
print("v2 deprecated? ", echo.deprecation)
print("v1 inputs:     ", list(echo_v1.inputs))
print("v2 inputs:     ", list(echo.inputs))

## Registering a package

```python
from myapp.nodes import math, text

registry = NodeRegistry()
registry.add_subregistry(text.registry)
registry.add_subregistry(math.registry)
```

Below, a throwaway package in a temporary directory.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

pkg = Path(tempfile.mkdtemp()) / "demo_nodes"
pkg.mkdir()
(pkg / "__init__.py").write_text("")
(pkg / "shout.py").write_text('''
from typing import Annotated

from conductor import NodeRegistry, Param
from conductor.widgets import TextInput

registry = NodeRegistry(category="text")


@registry.node("shout", title="Shout", description="Turns a text into capital letters and adds an exclamation mark.")
def shout(text: Annotated[str, Param(title="Input", widget=TextInput())]) -> str:
    return text.upper() + "!"
''')

sys.path.insert(0, str(pkg.parent))
shout = importlib.import_module("demo_nodes.shout")  # a package found at run time, as a host's plugins are

registry.add_subregistry(shout.registry)
print("registered:", [definition.id for definition in registry])

## The palette

`describe()` is a typed record: each field with its type's id and JSON schema, a widget as its `type` and options.

In [ ]:
import json

palette = registry.describe().nodes
print(json.dumps(palette[0].model_dump(mode="json"), indent=2))

## Summary

In [ ]:
print(f"Total nodes: {len(palette)}\n")
for entry in palette:
    print(f"  {entry.id}: {entry.title} (version {entry.version}; versions {entry.versions})")
    for inp in entry.inputs:
        optional = " (optional)" if not inp.required else ""
        widget = inp.widget["type"] if inp.widget else "none"
        print(f"      in:  {inp.name}: {inp.type} [{widget}]{optional}")
    for out in entry.outputs:
        print(f"      out: {out.name}: {out.type}")
    print()